In [0]:
storage_account_name = "storageenergy"
storage_account_access_key = "<YOUR_AZURE_ACCESS_KEY>"

spark.conf.set(f"fs.azure.account.key.{storage_account_name}.dfs.core.windows.net", storage_account_access_key)

silver_path = f"abfss://silver@{storage_account_name}.dfs.core.windows.net/telemetry_data/"

In [0]:
spark.read.format("delta").load(silver_path).createOrReplaceTempView("silver_telemetry")

In [0]:
%sql
CREATE OR REPLACE TEMP VIEW gold_energy_summary AS
SELECT 
    state,
    AVG(solar_irradiance) as avg_solar_potential,
    MAX(wind_speed) as peak_wind_speed,
    COUNT(*) as reading_count
FROM silver_telemetry
GROUP BY state

In [0]:
gold_df = spark.sql("SELECT * FROM gold_energy_summary")
gold_path = f"abfss://gold@{storage_account_name}.dfs.core.windows.net/energy_summary/"

gold_df.write.format("delta").mode("overwrite").save(gold_path)

In [0]:
gold_df.show()